In [1]:
import tensorflow as tf
import torch 
import tf2onnx
import onnx
import onnx2pytorch
from onnx2pytorch import ConvertModel

2025-03-24 00:08:37.818864: I tensorflow/core/util/port.cc:110] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-03-24 00:08:37.820090: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2025-03-24 00:08:37.841772: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2025-03-24 00:08:37.842807: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-03-24 00:08:39.554177: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT

In [2]:
tf.config.list_physical_devices("GPU")

2025-03-24 00:08:43.754609: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:996] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2025-03-24 00:08:43.757497: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:996] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2025-03-24 00:08:43.761259: W tensorflow/core/common_runtime/gpu/gpu_device.cc:1956] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the 

[]

In [4]:
# list available GPUs with torch
devices = [f"cuda:{i}" for i in range(torch.cuda.device_count())]
print("Available devices:", devices)

Available devices: ['cuda:0', 'cuda:1']


In [5]:
model = tf.keras.models.load_model(
    "../app/applications/deepfed/trained_models/dfer_model.h5"
)
# Convert the TensorFlow model to ONNX format


In [23]:
onnx_model, _ = tf2onnx.convert.from_keras(model)
pytorch_model = ConvertModel(onnx_model)

pytorch_model

2025-03-24 00:14:36.002055: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:996] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2025-03-24 00:14:36.015451: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:996] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2025-03-24 00:14:36.016852: I tensorflow/core/grappler/devices.cc:66] Number of eligible GPUs (core count >= 8, compute capability >= 0.0): 2
2025-03-24 00:14:36.016927: I tensorflow/core/grappler/clusters/single_machine.cc:358] Starting new session
2025-03-24 00:14:36.017174: I tensorflow/compiler/xla/stream_executor/

ConvertModel(
  (MatMul_model/dense/BiasAdd:0): Linear(in_features=13, out_features=32, bias=True)
  (Relu_model/dense/Relu:0): ReLU(inplace=True)
  (MatMul_model/dense_1/BiasAdd:0): Linear(in_features=32, out_features=16, bias=True)
  (Relu_model/dense_1/Relu:0): ReLU(inplace=True)
  (MatMul_model/dense_2/BiasAdd:0): Linear(in_features=16, out_features=7, bias=True)
  (Relu_model/dense_2/Relu:0): ReLU(inplace=True)
  (Reshape_model/conv2d_6/BiasAdd__84:0): Reshape(shape=[-1  1 96 96])
  (Conv_model/conv2d_6/BiasAdd:0): Sequential(
    (0): ConstantPad2d(padding=(1, 2, 1, 2), value=0)
    (1): Conv2d(1, 16, kernel_size=(6, 6), stride=(3, 3))
  )
  (Relu_model/conv2d_6/Relu:0): ReLU(inplace=True)
  (Conv_model/conv2d_7/BiasAdd:0): Conv2d(16, 32, kernel_size=(6, 6), stride=(2, 2), padding=(2, 2))
  (Relu_model/conv2d_7/Relu:0): ReLU(inplace=True)
  (Conv_model/conv2d_8/BiasAdd:0): Conv2d(32, 64, kernel_size=(6, 6), stride=(2, 2), padding=(2, 2))
  (Relu_model/conv2d_8/Relu:0): ReLU(inpla

In [7]:
model._get_save_spec()

[TensorSpec(shape=(None, 96, 96, 1), dtype=tf.float32, name='input_1'),
 TensorSpec(shape=(None, 96, 96, 1), dtype=tf.float32, name='input_2'),
 TensorSpec(shape=(None, 96, 96, 1), dtype=tf.float32, name='input_3'),
 TensorSpec(shape=(None, 13), dtype=tf.float32, name='input_4')]

In [32]:
input_1 = torch.randn(1, 96, 96, 1)
input_2 = torch.randn(1, 96, 96, 1)
input_3 = torch.randn(1, 96, 96, 1)
input_4 = torch.randn(1, 13)

output = pytorch_model([input_1, input_2, input_3, input_4])
print(output)

AttributeError: 'list' object has no attribute 'shape'

In [29]:
input_1 = tf.random.normal((100, 96, 96, 1))
input_2 = tf.random.normal((100, 96, 96, 1))
input_3 = tf.random.normal((100, 96, 96, 1))
input_4 = tf.random.normal((100, 13))

output = model([input_1, input_2, input_3, input_4])
print(output)

tf.Tensor(
[[1.78555950e-07 4.14850656e-05 9.98001635e-01 4.69046208e-04
  6.05049806e-08 2.56200031e-12 1.48759247e-03]
 [3.97913880e-08 1.41443852e-05 9.98974562e-01 1.80900402e-04
  1.60066449e-08 3.35237345e-13 8.30357661e-04]
 [9.55891096e-07 1.91039435e-04 9.97593462e-01 5.25471463e-04
  3.40405307e-07 4.38838722e-11 1.68864930e-03]
 [4.01954073e-03 2.48233732e-02 6.23268664e-01 6.68442994e-03
  3.15671525e-04 9.37371369e-05 3.40794563e-01]
 [5.43728383e-05 1.30394811e-03 9.88485456e-01 3.22431326e-03
  1.31790493e-05 8.40697645e-09 6.91881543e-03]
 [3.35520525e-07 8.71230732e-05 9.97772396e-01 4.77777445e-04
  1.61288028e-07 1.23992093e-11 1.66224816e-03]
 [7.14571343e-07 1.90191975e-04 9.98652041e-01 3.20810825e-04
  1.13760514e-07 1.44887644e-11 8.36172141e-04]
 [1.12115597e-08 3.73580042e-06 9.98427272e-01 1.02159986e-03
  2.39527100e-08 7.51438571e-14 5.47334785e-04]
 [7.49809828e-07 7.63618445e-05 9.95251298e-01 1.05001591e-03
  3.89129013e-07 4.71085046e-11 3.62110790e-03]

In [1]:
import requests

In [2]:
url = "https://api.github.com/users/willypaz243/repos"
res = requests.get(url)

In [3]:
repos = res.json()

In [5]:
for repo in repos:
    print(repo["name"])

activity_organizer
AgentesFinal
ANextG
api_cappuchino
BackendDevelopmentNodejs
cappuchino-parse
capuccino-schedule-extractor
CarServiceTBD
client_cappuchino
CrazyUno
DataScienceRoute
DeepFER
EC-API
economics-app
FaceAuth
Fingerprint-Enhancement-Python
GenericNestAPI
green_recovery_pix2pix_project
ivanapi
MineSweeperWithCS
ML_algorithms
Mushroom
page-web-ic
PracticaASO
product_catalog_api
Re-learning-git-and-github
roadmap-retos-programacion
rust_rpg_game
SearchEngineSportEvents
SocketIOExamples
